In [3]:
from typing import List, Tuple
from helper_class import MixtureMeta
import pandas as pd
from pathlib import Path
import os
import librosa
import numpy as np
from faster_whisper import WhisperModel

/home/jamin/mambaforge/envs/fw311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
faster_whiper_model = WhisperModel("large-v3", device="cuda", compute_type="int8")
segments, info = faster_whiper_model.transcribe(Path(os.pardir) / Path("Output/audio/mix_0001104_0.14_2_None_D.wav"))

In [ ]:
print(info)

TranscriptionInfo(language='en', language_probability=0.9892578125, duration=63.38775, duration_after_vad=63.38775, all_language_probs=[('en', 0.9892578125), ('cy', 0.006565093994140625), ('fr', 0.0004906654357910156), ('es', 0.0004229545593261719), ('pt', 0.0003819465637207031), ('de', 0.0003674030303955078), ('nn', 0.00022459030151367188), ('ru', 0.00021767616271972656), ('it', 0.00020611286163330078), ('la', 0.00019359588623046875), ('sv', 0.00018918514251708984), ('ja', 0.0001804828643798828), ('nl', 0.00014162063598632812), ('pl', 0.00011295080184936523), ('zh', 0.00011032819747924805), ('vi', 0.00010609626770019531), ('ko', 9.965896606445312e-05), ('tr', 7.176399230957031e-05), ('fi', 5.459785461425781e-05), ('ar', 4.601478576660156e-05), ('cs', 3.784894943237305e-05), ('da', 2.8789043426513672e-05), ('mi', 2.6226043701171875e-05), ('hi', 2.384185791015625e-05), ('no', 2.294778823852539e-05), ('hu', 1.8417835235595703e-05), ('id', 1.6927719116210938e-05), ('el', 1.662969589233398

TypeError: object of type 'Segment' has no len()

In [20]:
for segments in segments:
    print("start: {:.2f}s, end: {:.2f}s, text: {}".format(segments.start, segments.end, segments.text))

start: 0.00s, end: 2.72s, text:  While they went on holiday, we got the...
start: 2.72s, end: 5.74s, text:  They might be broken, but they are not defeated.
start: 7.04s, end: 10.02s, text:  Her condition was yesterday described as critical, but stable.
start: 10.02s, end: 11.36s, text:  His third goal was superb.
start: 12.62s, end: 14.70s, text:  This film will be totally awesome.
start: 16.32s, end: 19.26s, text:  I decided to tell a bit of the story about myself.
start: 19.78s, end: 21.10s, text:  I expect a rapid response.
start: 22.02s, end: 23.48s, text:  This is a national crisis.
start: 25.08s, end: 27.78s, text:  The figures are bad, but not that bad.
start: 27.78s, end: 30.62s, text:  It has all been arranged.
start: 30.62s, end: 33.32s, text:  Every time I play, I do not understand their reaction.
start: 35.18s, end: 37.16s, text:  It has moved on in the last week.
start: 39.74s, end: 41.98s, text:  Florida is a pivotal state in the nation.
start: 41.98s, end: 43.34s, text:

KeyboardInterrupt: 

In [4]:
def load_mixture_meta(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)

In [5]:
metas = load_mixture_meta(Path(os.pardir, Path("Output", "manifest.csv")))

In [6]:
def _split_into_two_chains(segments: List[Tuple[str, str]]) -> Tuple[List[str], List[str]]:
    """Split speaker-labeled segments into two ordered word chains."""
    # turn a string of segments into a list of tuples (speaker, word)
    segments = eval(segments)
    # print(f"Segments: {segments}")
    speakers = set(seg[0] for seg in segments)
    # print(f"Speakers: {speakers}")
    first_speaker = list(speakers)[0]
    # print(f"First speaker: {first_speaker}")
    chain_a = " ".join([seg[1] for seg in segments if seg[0] == first_speaker]).lower()
    second_speaker = list(speakers)[1]
    # print(f"Second speaker: {second_speaker}")
    chain_b = " ".join([seg[1] for seg in segments if seg[0] == second_speaker]).lower()
    return chain_a, chain_b

In [7]:
from wer_evaluation import get_hypothesis_transcription, load_asr_transcriptions
from mrs_beam_wer import mrs_wer_beam_2chain
#get clip id mix_0002525_0.40_2_7.4_T
# get corresponding transcription of parakeet from ASR_transcriptions.json
meta_test = metas[metas["clip_id"] == "mix_0000101_0.00_2_7.4_T"].iloc[0]
print(meta_test)
ref = meta_test["transcript"]

asr_data = load_asr_transcriptions(Path(os.pardir, "ASR_transcriptions.json"))
hyp = get_hypothesis_transcription(asr_data, "mix_0002525_0.40_2_7.4_T", "parakeet")
print(hyp)
print(ref)
chain_a, chain_b = _split_into_two_chains(ref)
print(chain_a)
print(chain_b)

clip_id                                          mix_0000101_0.00_2_7.4_T
audio_path                      Output/audio/mix_0000101_0.00_2_7.4_T.wav
transcript              [('218', 'A LAMP COULD NOT HAVE EXPIRED WITH M...
overlap_ratio_target                                                  0.0
overlap_ratio_actual                                                  0.0
max_speakers                                                            2
snr_db                                                                7.4
noise_type                                                              T
overlap_mask_path                                                 no mask
source_files            [PosixPath('/home/jamin/Year3Proj/Data/LibriSp...
noise_files                     [PosixPath('Data/Noise/TMETRO/ch10.wav')]
Name: 101, dtype: object
[('dummy-speaker', 'It is believed he had been stabbed. So you think you want to be Victoria? It is not I who desire it. It is the Lord, replied Savonarola coldl

In [43]:
mrs_wer_beam_2chain(chain_a, chain_b, hyp, 32, 1.0, True, True)

{'distance': 196,
 'wer': 1.0,
 'n_ref': 196,
 'alignment': [('delete_a', 'A', 'it', None),
  ('delete_a', 'A', 'is', None),
  ('delete_a', 'A', 'not', None),
  ('delete_a', 'A', 'i', None),
  ('delete_a', 'A', 'who', None),
  ('delete_a', 'A', 'desire', None),
  ('delete_a', 'A', 'it', None),
  ('delete_a', 'A', 'it', None),
  ('delete_a', 'A', 'is', None),
  ('delete_a', 'A', 'the', None),
  ('delete_a', 'A', 'lord', None),
  ('delete_a', 'A', 'replied', None),
  ('delete_a', 'A', 'savonarola', None),
  ('delete_a', 'A', 'coldly', None),
  ('delete_a', 'A', 'impossible', None),
  ('delete_a', 'A', 'impossible', None),
  ('delete_a', 'A', 'murmured', None),
  ('delete_a', 'A', 'lorenzo', None),
  ('delete_a', 'A', 'very', None),
  ('delete_a', 'A', 'well', None),
  ('delete_a', 'A', 'then', None),
  ('delete_a', 'A', 'die', None),
  ('delete_a', 'A', 'as', None),
  ('delete_a', 'A', 'you', None),
  ('delete_a', 'A', 'have', None),
  ('delete_a', 'A', 'lived', None),
  ('delete_a', 'A'

In [ ]:
# extract row whose transcript contains ("any speaker", None) or ("any speaker", "None") or ("any speaker", np.nan) where transcript is a list of tuples (speaker, transcript)
row_none = metas[metas["transcript"].apply(lambda x: any((t[0] == "any speaker" and (t[1] is None or t[1] == "None" or (isinstance(t[1], float) and np.isnan(t[1]))) for t in eval(x))))]


In [ ]:
row_none

,clip_id,audio_path,transcript,overlap_ratio_target,overlap_ratio_actual,max_speakers,snr_db,noise_type,overlap_mask_path,source_files,noise_files


In [ ]:
import os
print(os.environ.get("LD_LIBRARY_PATH"))

/home/jamin/mambaforge/envs/fw311/lib/python3.11/site-packages/nvidia/cublas/lib:/home/jamin/mambaforge/envs/fw311/lib/python3.11/site-packages/nvidia/cudnn/lib:


In [ ]:
import ctypes
ctypes.CDLL("libcublas.so.12")
ctypes.CDLL("libcudnn.so.9")
print("CUDA libs load OK")

CUDA libs load OK


In [ ]:
def load_mixture_meta(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)
def load_mixture_audio(path: Path, sr: int = 16000) -> Tuple[np.ndarray, int]:
    audio, sr = librosa.load(path, sr=sr)
    return audio, sr

In [ ]:
meta_df = load_mixture_meta(Path(os.pardir,"Output", "manifest.csv"))

In [ ]:
meta_df.clip_id.filter

<bound method NDFrame.filter of 0       mix_0.00_2_None_D_0000000
1       mix_0.00_2_None_P_0000001
2       mix_0.00_2_None_T_0000002
3        mix_0.00_2_7.4_D_0000003
4        mix_0.00_2_7.4_P_0000004
                  ...            
3715       mix_0.40_2_0_P_0003715
3716       mix_0.40_2_0_T_0003716
3717      mix_0.40_2_-5_D_0003717
3718      mix_0.40_2_-5_P_0003718
3719      mix_0.40_2_-5_T_0003719
Name: clip_id, Length: 3720, dtype: str>

In [ ]:
# filter out clip ids that starts with mix_0.14
test_meta = meta_df[meta_df.clip_id.str.startswith("mix_0.14_2_None_D_0000144")].head(1)
test_meta

,clip_id,audio_path,transcript,overlap_ratio_target,overlap_ratio_actual,max_speakers,snr_db,noise_type,overlap_mask_path,source_files,noise_files
144,mix_0.14_2_None_D_0000144,Output/audio/mix_0.14_2_None_D_0000144.wav,"[('125-121342-0038', ""THEY DON'T THEY JUST WAN...",0.14,0.14,2,NaN,D,Output/masks/mix_0.14_2_None_D_0000144.npy,[PosixPath('/home/jamin/Year3Proj/Data/LibriSp...,[]


In [ ]:
print(test_meta.audio_path.values[0])
print(Path(os.pardir, test_meta.audio_path.values[0]))

Output/audio/mix_0.14_2_None_D_0000144.wav
../Output/audio/mix_0.14_2_None_D_0000144.wav


In [ ]:
# model = WhisperModel("large-v3", device="cuda")

# segments, info = model.transcribe(Path(os.pardir, test_meta.audio_path.values[0]))
# for segment in segments:
#     print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))


In [ ]:
# import torch
# from transformers import pipeline

# device = 0 if torch.cuda.is_available() else -1

# asr = pipeline(
#     "automatic-speech-recognition",
#     model="facebook/wav2vec2-large-960h",
#     device=0,
# )
# audio = load_mixture_audio(Path(os.pardir, test_meta.audio_path.values[0]))[0]
# result = asr(audio)
# print(result["text"])

In [ ]:
# from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
# import torch

# # load model and processor
# processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
# model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h")
    
# # load dummy dataset and read soundfiles
# audio = load_mixture_audio(Path(os.pardir, test_meta.audio_path.values[0]))[0]

# # tokenize
# input_values = processor(audio, return_tensors="pt", padding="longest").input_values  # Batch size 1

# # retrieve logits
# logits = model(input_values).logits

# # take argmax and decode
# predicted_ids = torch.argmax(logits, dim=-1)
# transcription = processor.batch_decode(predicted_ids)


In [ ]:
# print(test_meta.transcript.values[0].astype(str))

# print(" ".join(seg[1] for seg in test_meta.transcript.values[0]))

In [ ]:
# import nemo.collections.asr as nemo_asr
# asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v3")
# output = asr_model.transcribe([Path(os.pardir, test_meta.audio_path.values[0])])
# print(output[0].text)


In [ ]:
# import whisperx
# device = "cuda"   # or "cpu"
# batch_size = 1
# compute_type = "int8"
# model = whisperx.load_model("large-v2", device, compute_type=compute_type)
# audio = whisperx.load_audio(Path(os.pardir) / test_meta.audio_path.values[0])

# result = model.transcribe(audio, batch_size=batch_size)

In [ ]:
# import nemo.collections.asr as nemo_asr
# asr_model = nemo_asr.models.ASRModel.from_pretrained("nvidia/parakeet-tdt-0.6b-v2")
# transcript = asr_model.transcribe([test_meta.audio_path.values[0]])

[NeMo W 2026-04-04 00:15:05 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
import pickle
with open("funasr_output.pkl", "rb") as f:
    funasr_output = pickle.load(f)


In [ ]:
funasr_output[0][0]

{'key': 'rand_key_1t9EwL56nGisi',
 'text': "They don't. They just want the fun of eating it all over again. The matron doesn't want to repeat her own feelings, but she's just a part of the world. Again, Ventnor's rifle cracked. One of the colors of the world is down, another's down, more did he reproach himself for feelings that were bound to the horseman, but his sworn foot soldiers, a forest of shining planks above them, clearly to us came their battle cries. Again, Ventnor's rifle cracked. I detest poor people. I hate them for being poor. Poverty may have been beautiful for the horseman, but it's one foot in a forest and truly a big faint radiance. It was her eyes, her great wide eyes whose clear form magnified us. Over us whined the bullets from the covering guns. The fathom deeper by lie, with old desires restrained before. No, I heard a song. The matron doesn't want to repeat her. We just want to repeat her honeymoon. How the smell of it suddenly cleared the air and threw everyon